# Voice Detection — Dataset & DataLoader

Validates the `SpeakerChunkDataset` class: item shapes, value ranges, speaker counts, and `DataLoader` batch throughput.

## 1. Imports

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

sys.path.insert(0, str(Path("..").resolve()))

from voice_detection.dataset import SpeakerChunkDataset

## 2. Config

In [ ]:
ROOT          = Path("..").resolve()
MANIFEST_PATH = ROOT / "data/manifest.csv"

# Mel spectrogram parameters — must match what you want to feed the model
N_MELS      = 80
N_FFT       = 400    # 25 ms window @ 16 kHz
HOP_LENGTH  = 160    # 10 ms hop @ 16 kHz
SAMPLE_RATE = 16_000
NORMALIZE   = True   # per-chunk mean/std normalisation

BATCH_SIZE  = 32
NUM_WORKERS = 0      # set >0 on Linux/macOS; keep 0 on Windows to avoid spawn issues

## 3. Instantiate datasets for all splits

In [ ]:
datasets = {}
for split in ("train", "val", "test"):
    ds = SpeakerChunkDataset(
        manifest_path=MANIFEST_PATH,
        split=split,
        n_mels=N_MELS,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        sample_rate=SAMPLE_RATE,
        normalize=NORMALIZE,
    )
    datasets[split] = ds
    print(f"{split:<6}  {len(ds):>8,} chunks   {ds.num_speakers:>4} speakers")

## 4. Inspect a single item

Check the spectrogram tensor shape and value range.

In [ ]:
train_ds = datasets["train"]
spec, label = train_ds[0]

print(f"Spectrogram shape : {tuple(spec.shape)}  (C, n_mels, T_frames)")
print(f"dtype             : {spec.dtype}")
print(f"value range       : [{spec.min():.3f}, {spec.max():.3f}]")
print(f"Speaker label     : {label}")

# Plot it
fig, ax = plt.subplots(figsize=(10, 3))
img = ax.imshow(spec.squeeze(0).numpy(), origin="lower", aspect="auto", cmap="magma")
ax.set_title("Log-Mel Spectrogram — train[0]")
ax.set_xlabel("Time frames")
ax.set_ylabel("Mel bins")
plt.colorbar(img, ax=ax)
plt.tight_layout()
plt.show()

## 5. DataLoader batch test

Verify that the `DataLoader` produces correctly shaped batches.

In [ ]:
train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
)

batch_specs, batch_labels = next(iter(train_loader))
print(f"Batch spectrogram shape : {tuple(batch_specs.shape)}  (B, C, n_mels, T_frames)")
print(f"Batch labels shape      : {tuple(batch_labels.shape)}")
print(f"Unique labels in batch  : {batch_labels.unique().tolist()}")

## 6. Speaker label distribution

Chunk counts per speaker in the training set — useful for spotting class imbalance.

In [ ]:
manifest = pd.read_csv(MANIFEST_PATH)
train_manifest = manifest[manifest["split"] == "train"]

chunks_per_speaker = (
    train_manifest.groupby(["dataset", "speaker_id"])
    .size()
    .reset_index(name="chunks")
    .sort_values("chunks", ascending=False)
)

print(f"Training speakers: {len(chunks_per_speaker)}")
print(f"Chunks per speaker — min: {chunks_per_speaker['chunks'].min()}  "
      f"median: {chunks_per_speaker['chunks'].median():.0f}  "
      f"max: {chunks_per_speaker['chunks'].max()}")

fig, ax = plt.subplots(figsize=(10, 3))
ax.hist(chunks_per_speaker["chunks"], bins=50, edgecolor="none")
ax.set_title("Distribution of chunks per speaker (train)")
ax.set_xlabel("Chunks")
ax.set_ylabel("Number of speakers")
plt.tight_layout()
plt.show()